In [ ]:
WITH acct AS  
(
    SELECT 
        account_id AS salesforce_account_id,
        account_nm AS salesforce_account_name,
        became_customer_dt AS date_became_customer,
        territory_sgmnt_nm AS sales_segment,
        churn_rsk_ind AS churn_risk_input_by_cs,
        account_typ_nm,
        new_global_ee_segmt_override_nm,
        new_global_segmt_override_nm
    FROM PROD_HARMONIZED.SALES.VW_ACCOUNT_PROF
    WHERE crnt_rcrd_ind = 'Y'
),

prod_tenant_info AS
(
    SELECT
        CONCAT(data_cntr_cd,'.',tenant_id) AS vh_tenant_id,
        tenant_id,
        tenant_nm,
        account_id,
        data_cntr_cd
    FROM PROD_HARMONIZED.PRODUCT.TENANT_PROF
    WHERE crnt_rcrd_ind = 'Y'
      AND tenant_typ_nm = 'PRODUCTION'
      AND organization_typ_nm = 'CUSTOMER'
      AND tenant_enbld_status_nm = 'ENABLED'
      AND tenant_nm NOT ILIKE '%spark%'
      AND tenant_nm NOT ILIKE '%demo%'
),

user_summary AS
(
    SELECT
        tenant_dc AS vh_tenant_id,
        COUNT(DISTINCT user_id) AS total_users_enabled,
        COUNT(DISTINCT CASE WHEN copilot_access_ind = 'Y' THEN user_id END) AS total_vcp_users_enabled,
        COUNT(DISTINCT CASE WHEN admin_access_ind = 'Y' OR mdlr_access_ind = 'Y' OR mngr_access_ind = 'Y' THEN user_id END) AS no_power_users,
        COUNT(DISTINCT CASE WHEN (admin_access_ind = 'Y' OR mdlr_access_ind = 'Y' OR mngr_access_ind = 'Y')
                              AND copilot_access_ind = 'Y' THEN user_id END) AS no_vcp_power_users,
        COUNT(DISTINCT CASE WHEN admin_access_ind = 'N'
                              AND mdlr_access_ind = 'N'
                              AND mngr_access_ind = 'N'
                              AND cntrbtr_access_ind = 'Y' THEN user_id END) AS no_business_users,
        COUNT(DISTINCT CASE WHEN admin_access_ind = 'N'
                              AND mdlr_access_ind = 'N'
                              AND mngr_access_ind = 'N'
                              AND cntrbtr_access_ind = 'Y'
                              AND copilot_access_ind = 'Y' THEN user_id END) AS no_vcp_business_users,
        COUNT(DISTINCT CASE WHEN viewer_access_ind = 'Y' THEN user_id END) AS no_view_only,
        COUNT(DISTINCT CASE WHEN admin_access_ind = 'N'
                              AND mdlr_access_ind = 'N'
                              AND mngr_access_ind = 'N'
                              AND cntrbtr_access_ind = 'N'
                              AND dshbrdr_access_ind = 'Y' THEN user_id END) AS no_dashboarder_only
    FROM PROD_HARMONIZED.PRODUCT.VW_USER_PROF_EML_INCLD
    WHERE crnt_rcrd_ind = 'Y'
      AND actvtn_status_nm = 'ACTIVE'
      AND built_in_admin_accnt = 'N'
    GROUP BY tenant_dc
),

arr_usd AS
(
    SELECT
        dc_tenant_id AS vh_tenant_id,
        arr_usd
    FROM PROD_HARMONIZED.PRODUCT.VW_ENBLD_PRODUCTION_TENANTS
),

process AS
(
    SELECT 
        CONCAT(data_cntr_cd,'.',tenant_id) AS vh_tenant_id,
        MAX(CASE WHEN process_status_nm = 'COMPLETED' THEN eff_dts END) AS latest_process_run_complete,
        COUNT(DISTINCT CASE WHEN process_status_nm <> 'DELETED' THEN process_id END) AS no_total_processes_ltm,
        COUNT(DISTINCT CASE WHEN process_status_nm = 'COMPLETED' THEN eff_dts END) AS no_process_runs_completed_ltm
    FROM PROD_HARMONIZED.PRODUCT.TENANT_PROCESS_LOG
    WHERE eff_dts >= CURRENT_DATE() - 365
    GROUP BY 1
),

etl_steps AS
(
    SELECT
        CONCAT(data_center,'.',tenant_id) AS vh_tenant_id,
        COUNT(DISTINCT job_id) AS jobs_ltm_steps
    FROM PROD_RAW.MT_SERVER.MTSERVER_ETL_STEPS
    WHERE TO_DATE(started_ts) >= CURRENT_DATE() - 365
    GROUP BY 1
),

etl_jobs AS
(
    SELECT
        CONCAT(data_center,'.',tenant_id) AS vh_tenant_id,
        COUNT(DISTINCT id) AS jobs_ltm_ui
    FROM PROD_RAW.MT_SERVER.MTSERVER_ETL_JOBS
    WHERE TO_DATE(created_date) >= CURRENT_DATE() - 365
    GROUP BY 1
),

model_dim_ranked AS
(
    SELECT
        tenant_id,
        data_cntr_cd,
        CONCAT(data_cntr_cd,'.',tenant_id) AS vh_tenant_id,
        model_id,
        dim_id,
        dim_nm,
        etl_updt_dts,
        MAX(etl_updt_dts) OVER (PARTITION BY data_cntr_cd, tenant_id) AS latest_etl
    FROM PROD_HARMONIZED.PRODUCT.MODEL_DIM_DTL
),

latest_model_dims AS
(
    SELECT *
    FROM model_dim_ranked
    WHERE etl_updt_dts = latest_etl
),

latest_active_dim_members AS
(
    SELECT
        tenant_id,
        data_cntr_cd,
        dim_id,
        member_id
    FROM PROD_HARMONIZED.PRODUCT.DIM_MEMBER_DTL
    WHERE schdld_delete_dts IS NULL
),

model_dim_counts AS
(
    SELECT
        vh_tenant_id,
        COUNT(DISTINCT model_id) AS no_models_created,
        COUNT(DISTINCT dim_id) AS no_dimensions
    FROM latest_model_dims
    GROUP BY vh_tenant_id
),

member_counts AS
(
    SELECT
        lmd.vh_tenant_id,
        COUNT(DISTINCT ladm.member_id) AS no_members,
        COUNT(DISTINCT CASE WHEN lmd.dim_nm ILIKE '%measure%' THEN ladm.member_id END) AS members_for_measure_dim
    FROM latest_model_dims lmd
    LEFT JOIN latest_active_dim_members ladm
        ON lmd.tenant_id = ladm.tenant_id
       AND lmd.data_cntr_cd = ladm.data_cntr_cd
       AND lmd.dim_id = ladm.dim_id
    GROUP BY lmd.vh_tenant_id
),

model_stats AS
(
    SELECT
        mdc.vh_tenant_id,
        mdc.no_models_created,
        mdc.no_dimensions,
        mc.no_members,
        mc.members_for_measure_dim
    FROM model_dim_counts mdc
    LEFT JOIN member_counts mc
        ON mdc.vh_tenant_id = mc.vh_tenant_id
),

most_members_per_dim AS
(
    SELECT
        vh_tenant_id,
        MAX(cnt) AS most_members_single_dimension
    FROM
    (
        SELECT
            lmd.vh_tenant_id,
            lmd.dim_id,
            COUNT(ladm.member_id) AS cnt
        FROM latest_model_dims lmd
        LEFT JOIN latest_active_dim_members ladm
            ON lmd.tenant_id = ladm.tenant_id
           AND lmd.data_cntr_cd = ladm.data_cntr_cd
           AND lmd.dim_id = ladm.dim_id
        GROUP BY 1, 2
    ) x
    GROUP BY vh_tenant_id
),

vtable AS
(
    SELECT
        CONCAT(data_center,'.',tenant_id) AS vh_tenant_id,
        MAX(row_count) AS row_count_largest_vtable,
        MAX(size_in_bytes) / POWER(1024, 3) AS size_largest_vtable_gb
    FROM PROD_RAW.MT_SERVER.MTSERVER_VENA_TABLES_STAGING
    GROUP BY 1
),

base_output AS
(
    SELECT
        p.vh_tenant_id,
        p.tenant_nm,
        p.account_id,

        a.salesforce_account_name,
        a.date_became_customer,
        a.sales_segment,
        a.churn_risk_input_by_cs,
        a.new_global_ee_segmt_override_nm,
        a.new_global_segmt_override_nm,

        ar.arr_usd,

        us.total_users_enabled,
        us.total_vcp_users_enabled,
        us.no_power_users,
        us.no_vcp_power_users,
        us.no_business_users,
        us.no_vcp_business_users,
        us.no_view_only,
        us.no_dashboarder_only,

        ms.no_models_created,
        ms.no_dimensions,
        ms.no_members,

        mmpd.most_members_single_dimension,
        ms.members_for_measure_dim,

        pr.latest_process_run_complete,
        pr.no_total_processes_ltm,
        pr.no_process_runs_completed_ltm,

        es.jobs_ltm_steps,
        ej.jobs_ltm_ui,

        vt.row_count_largest_vtable,
        vt.size_largest_vtable_gb
    FROM prod_tenant_info p
    LEFT JOIN acct a
        ON LEFT(p.account_id, 15) = LEFT(a.salesforce_account_id, 15)
    LEFT JOIN arr_usd ar
        ON ar.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN user_summary us
        ON us.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN model_stats ms
        ON ms.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN most_members_per_dim mmpd
        ON mmpd.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN process pr
        ON pr.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN etl_steps es
        ON es.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN etl_jobs ej
        ON ej.vh_tenant_id = p.vh_tenant_id
    LEFT JOIN vtable vt
        ON vt.vh_tenant_id = p.vh_tenant_id
    WHERE NOT (
        p.tenant_nm ILIKE '%wells fargo%'
        OR p.tenant_nm ILIKE '%state street%'
        OR p.tenant_nm ILIKE '%abbott%'
    )
),

arr_usd_pct AS
(
    SELECT
        vh_tenant_id,
        CUME_DIST() OVER (ORDER BY arr_usd) AS arr_usd_pct
    FROM base_output
    WHERE arr_usd IS NOT NULL
),

no_models_pct AS
(
    SELECT
        vh_tenant_id,
        CUME_DIST() OVER (ORDER BY no_models_created) AS no_models_pct
    FROM base_output
    WHERE no_models_created IS NOT NULL
),

no_dimensions_pct AS
(
    SELECT
        vh_tenant_id,
        CUME_DIST() OVER (ORDER BY no_dimensions) AS no_dimensions_pct
    FROM base_output
    WHERE no_dimensions IS NOT NULL
),

no_members_pct AS
(
    SELECT
        vh_tenant_id,
        CUME_DIST() OVER (ORDER BY no_members) AS no_members_pct
    FROM base_output
    WHERE no_members IS NOT NULL
),

jobs_steps_pct AS
(
    SELECT
        vh_tenant_id,
        CUME_DIST() OVER (ORDER BY jobs_ltm_steps) AS jobs_steps_pct
    FROM base_output
    WHERE jobs_ltm_steps IS NOT NULL
),

jobs_ui_pct AS
(
    SELECT
        vh_tenant_id,
        CUME_DIST() OVER (ORDER BY jobs_ltm_ui) AS jobs_ui_pct
    FROM base_output
    WHERE jobs_ltm_ui IS NOT NULL
)

SELECT
    b.vh_tenant_id,
    b.tenant_nm,
    b.account_id,

    b.salesforce_account_name,
    b.date_became_customer,
    b.sales_segment,
    b.churn_risk_input_by_cs,
    b.new_global_ee_segmt_override_nm,
    b.new_global_segmt_override_nm,

    b.arr_usd,
    aup.arr_usd_pct,

    b.total_users_enabled,
    b.total_vcp_users_enabled,
    b.no_power_users,
    b.no_vcp_power_users,
    b.no_business_users,
    b.no_vcp_business_users,
    b.no_view_only,
    b.no_dashboarder_only,

    b.no_models_created,
    nmp.no_models_pct,

    b.no_dimensions,
    ndp.no_dimensions_pct,

    b.no_members,
    nmem.no_members_pct,

    b.most_members_single_dimension,
    b.members_for_measure_dim,

    b.latest_process_run_complete,
    b.no_total_processes_ltm,
    b.no_process_runs_completed_ltm,

    b.jobs_ltm_steps,
    jsp.jobs_steps_pct,

    b.jobs_ltm_ui,
    jup.jobs_ui_pct,

    b.row_count_largest_vtable,
    b.size_largest_vtable_gb
FROM base_output b
LEFT JOIN arr_usd_pct aup
    ON aup.vh_tenant_id = b.vh_tenant_id
LEFT JOIN no_models_pct nmp
    ON nmp.vh_tenant_id = b.vh_tenant_id
LEFT JOIN no_dimensions_pct ndp
    ON ndp.vh_tenant_id = b.vh_tenant_id
LEFT JOIN no_members_pct nmem
    ON nmem.vh_tenant_id = b.vh_tenant_id
LEFT JOIN jobs_steps_pct jsp
    ON jsp.vh_tenant_id = b.vh_tenant_id
LEFT JOIN jobs_ui_pct jup
    ON jup.vh_tenant_id = b.vh_tenant_id
;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from kneed import KneeLocator

# df should have one row per customer
# example columns:
# CUSTOMER_ID, COL_A, COL_B, COL_C

metric_cols = ["no_models_created", "arr_usd", "jobs_ltm_steps"]

results = []

for col in metric_cols:
    # keep only valid rows for this metric
    temp = (
        df[["vh_tenant_id", col]]
        .dropna()
        .sort_values(col)
        .reset_index(drop=True)
    )

    # rank becomes x
    temp["RANK"] = np.arange(1, len(temp) + 1)

    x = temp["RANK"].values
    y = temp[col].values

    kl = KneeLocator(
        x,
        y,
        curve="convex",
        direction="increasing"
    )

    knee_rank = kl.knee
    knee_value = kl.knee_y

    if knee_rank is not None:
        knee_customer = temp.loc[temp["RANK"] == knee_rank, "vh_tenant_id"].iloc[0]
    else:
        knee_customer = None

    results.append({
        "METRIC": col,
        "KNEE_RANK": knee_rank,
        "KNEE_VALUE": knee_value,
        "KNEE_CUSTOMER_ID": knee_customer
    })

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(x, y)
    if knee_rank is not None:
        plt.axvline(knee_rank, linestyle="--")
        plt.scatter([knee_rank], [knee_value], s=60)
    plt.title(f"Knee Detection for {col}")
    plt.xlabel("Customer Rank")
    plt.ylabel(col)
    plt.show()

results_df = pd.DataFrame(results)
results_df

In [ ]:
#import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = customer_data.copy()
df.columns = df.columns.str.strip()

cols = ["NO_MODELS_CREATED", "JOBS_LTM_STEPS"]

def find_knee(y_values):
    y = np.sort(np.array(y_values, dtype=float))
    n = len(y)

    if n < 3:
        return None, None, None

    x = np.arange(n, dtype=float)

    # points
    points = np.column_stack((x, y))

    # line from first to last point
    start = points[0]
    end = points[-1]
    line_vec = end - start
    line_len = np.linalg.norm(line_vec)

    if line_len == 0:
        return None, None, None

    # perpendicular distance from each point to the line
    distances = np.abs(
        line_vec[1] * (points[:, 0] - start[0]) -
        line_vec[0] * (points[:, 1] - start[1])
    ) / line_len

    knee_index = int(np.argmax(distances))
    knee_value = y[knee_index]
    percentile = (knee_index + 1) / n

    return knee_index, knee_value, percentile

def plot_knee(y_values, col_name):
    y = np.sort(np.array(y_values, dtype=float))
    x = np.arange(len(y))

    result = find_knee(y)
    if result[0] is None:
        print(f"No knee found for {col_name}")
        return

    knee_idx, knee_val, pct = result

    plt.figure(figsize=(8, 5))
    plt.plot(x, y)
    plt.axvline(knee_idx, linestyle="--")
    plt.scatter([knee_idx], [knee_val], s=60)
    plt.title(f"Knee Detection: {col_name}")
    plt.xlabel("Customer Rank")
    plt.ylabel(col_name)
    plt.show()

    print(f"{col_name}")
    print(f"  Knee rank: {knee_idx}")
    print(f"  Knee value: {knee_val}")
    print(f"  Percentile: {pct:.2%}")

results = []

for col in cols:
    if col not in df.columns:
        print(f"Missing column: {col}")
        continue

    data = pd.to_numeric(df[col], errors="coerce").dropna().values

    if len(data) == 0:
        print(f"No numeric data in {col}")
        continue

    knee_idx, knee_val, pct = find_knee(data)

    results.append({
        "metric": col,
        "knee_rank": knee_idx,
        "knee_value": knee_val,
        "percentile": pct,
        "% above knee": None if pct is None else 1 - pct
    })

    plot_knee(data, col)

results_df = pd.DataFrame(results)
display(results_df)